# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the [`mlcroissant`](https://mlcommons.org/croissant/) library.

We use each entity's `@id` for reference to ensure full reproducibility and traceability to the Croissant schema.

### Dataset Source
- [Croissant Schema JSON-LD](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json)
- Citation: Mugotitsa, B, Maina, D, Owoko, H, Akinyi, L, Greenfield, J, Todd, J, Kavu, T and Kiragga, A 2026 A FAIR2 Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Frontiers.

In [ ]:
# Ensure the mlcroissant library is installed
!pip install mlcroissant --quiet

## 1. Data Loading

We load the Croissant dataset's metadata and inspect the high-level information about the resource before attempting to access records.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata
print(f"{meta.name}: {meta.description}\n")
print(f"Published: {getattr(meta, 'datePublished', 'N/A')}")
print(f"Version: {getattr(meta, 'version', 'N/A')}")

## 2. Data Overview

Explore available record sets and their associated fields. All objects are referenced by their `@id` as per the Croissant schema.

Below, we print all available record sets and their associated fields (with `@id` and `name`).

In [ ]:
# List all record sets, fields, and columns using their @id as reference
record_sets = list(dataset.record_sets)
print(f"Total record sets: {len(record_sets)}")
rs_meta = {}
for rs in record_sets:
    print(f"\nRecord Set Name: {rs.name}")
    print(f"  @id: {rs.id}")
    field_ids = []
    for field in rs.fields:
        print(f"    Field: {field.name} (@id: {field.id})")
        if hasattr(field, 'columns'):
            for col in field.columns:
                print(f"      Column: {col.name} (@id: {col.id}) [{col.data_type}]")
        field_ids.append(field.id)
    rs_meta[rs.id] = field_ids

## 3. Data Extraction

We'll extract data from each record set (using its `@id`) and load it into a Pandas DataFrame for downstream analysis. The columns correspond to their field `@id` or field names.

In [ ]:
# Extract records for each record set by @id and load into a Pandas DataFrame
dfs = {}
record_set_ids = [rs.id for rs in dataset.record_sets]
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dfs[rs_id] = df
    print(f"Loaded {len(df)} records for record_set @id: {rs_id}")

# Choose the main record set for demonstration. Often, the main data table is the first record set.
main_rs_id = record_set_ids[0] if record_set_ids else None

print("\nColumns in the primary record set:")
print(list(dfs[main_rs_id].columns))
dfs[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)

Demonstrate filtering, normalization, and group-by operations on a numeric field. We'll also show how to use `@id` to refer to a particular numeric and group fields, based on the overview above.

Let's use the `phq9_score` field (if present, as indicated by the dataset's context of containing PHQ-9 scores) and group by `'gender'` or its corresponding `@id`.

In [ ]:
main_df = dfs[main_rs_id]
# Try to find likely numeric fields (example: phq9_score, gad7_score, psq_score). List columns to decide.
possible_num_fields = [col for col in main_df.columns if any(s in col.lower() for s in ['phq9', 'gad7', 'psq', 'score', 'total'])]
print(f"Possible numeric fields: {possible_num_fields}")
numeric_field_id = possible_num_fields[0] if possible_num_fields else None

# Try to guess a good group field (e.g., gender, education, marital_status)
possible_group_fields = [col for col in main_df.columns if any(s in col.lower() for s in ['gender', 'education', 'marital', 'village'])]
group_field_id = possible_group_fields[0] if possible_group_fields else None

if numeric_field_id and group_field_id:
    print(f"Using field @id '{numeric_field_id}' as numeric, and '{group_field_id}' as group field.")

    # Remove missing values for simplicity
    edf = main_df.dropna(subset=[numeric_field_id])

    # Define a threshold at the mean for demonstration
    threshold = edf[numeric_field_id].mean()
    filtered_df = edf[edf[numeric_field_id] > threshold]
    print(f"\nFiltered {len(filtered_df)} records where {numeric_field_id} > {threshold:.2f}")

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by group_field
    grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(f"\nMean {numeric_field_id} by {group_field_id}:")
    print(grouped)
else:
    print("Could not automatically detect appropriate numeric or group fields. Please review dataset column list above.")

## 5. Visualization

Plot distributions or comparisons of key numeric fields, grouped by a categorical field (all referenced by their `@id`).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and group_field_id:
    plt.figure(figsize=(8, 5))
    sns.boxplot(data=main_df, x=group_field_id, y=numeric_field_id)
    plt.title(f"{numeric_field_id} distribution by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

    # Histogram
    plt.figure(figsize=(6, 4))
    sns.histplot(data=main_df, x=numeric_field_id, bins=15, kde=True, color='teal')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

## 6. Conclusion

In this notebook, we demonstrated how to systematically explore and analyze a Croissant dataset using the `mlcroissant` library. By referencing entities explicitly via their `@id`, we ensured provenance and reproducibility in every step. This approach can be adapted to any Croissant-compatible data package for AI-ready data exploration.

The dataset provides a valuable snapshot of mental health indicators in Kilifi County, Kenya, and can be further filtered, grouped, and visualized for scientific and public health insights.